## Question 7 — Softmax stability and temperature

A direct softmax overflows when a score is large, because $\exp(1000)$ is not a finite float. Subtracting the maximum score before the exponential leaves the probabilities unchanged. Temperature $T$ divides the scores first: $T<1$ sharpens the weights and $T>1$ flattens them toward uniform.

**Imports.** NumPy computes both softmaxes. Matplotlib draws one curve per temperature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

**Stable softmax.** Subtract the maximum along the softmax axis, then exponentiate. The shift cancels in the normalization, so the weights match a direct softmax when that softmax is finite.

In [ ]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)

**Unstable softmax.** $\exp(1000)$, $\exp(1001)$, and $\exp(1002)$ overflow, so these weights are not finite.

In [ ]:
large = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    direct = np.exp(large) / np.sum(np.exp(large))
print("Q7 direct:", direct)

**Stable result.** The same scores stay finite. The largest score receives almost all of the weight, and the weights sum to $1$.

In [ ]:
print("Q7 stable:", stable_softmax(large))
assert not np.all(np.isfinite(direct))
np.testing.assert_allclose(stable_softmax(large).sum(), 1)
print("Stability check passed.")

**Temperature weights.** The scores are the $2\times 2$ identity. Each temperature divides those scores before the row-wise softmax.

In [ ]:
temperatures = [0.5, 1.0, 2.0, 4.0]
raw_scores = np.eye(2)
temperature_weights = {
    T: stable_softmax(raw_scores / T, axis=-1) for T in temperatures
}

**First query.** Key $0$ matches this query. A smaller $T$ puts more probability on that key.

In [ ]:
for T, A in temperature_weights.items():
    print(f"T={T}: first query weights={A[0]}")

**Plot.** Each curve is the first query's attention over the two keys.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for T, A in temperature_weights.items():
    ax.plot([0, 1], A[0], marker="o", label=f"T={T:g}")
ax.set(
    xlabel="Key index",
    ylabel="Attention probability",
    title="Attention temperature: first query",
    xticks=[0, 1],
    ylim=(0, 1),
)
ax.legend()
plt.show()